# Panel Unit Root Tests: Detecting Non-Stationarity in Panel Data

**Version**: 1.0  
**Date**: 2026-02-22  
**Estimated Duration**: 90 minutes  
**Difficulty**: Intermediate

## Learning Objectives
1. Understand the difference between stationary and non-stationary (I(1)) time series
2. Explain why unit roots matter for panel data regression (spurious regression problem)
3. Apply multiple panel unit root tests: IPS, LLC, Breitung, and Hadri
4. Interpret test results, especially when tests disagree
5. Decide whether to use levels or first differences based on test results
6. Use the unified interface `panel_unit_root_test()` for comparative analysis
7. Recognize the reversed null hypothesis in Hadri test (H0: stationarity)

## Prerequisites
- Basic understanding of time series concepts (AR processes)
- Familiarity with panel data structure
- Completion of static panel models tutorials

## Dataset
Penn World Table (simulated), Regional Prices, State Unemployment

## Estimated Time per Section
| Section | Topic | Time |
|---------|-------|------|
| 1 | What is a Unit Root? | 15 min |
| 2 | Panel Unit Root Tests (theory) | 15 min |
| 3 | Practical Application | 20 min |
| 4 | Unified Interface | 10 min |
| 5 | Visualization | 5 min |
| 6 | Decision Making | 10 min |
| 7 | Key Takeaways | 5 min |
| Exercises | Practice | 10 min |

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Local utilities
import sys

import statsmodels.api as sm

# PanelBox imports
import panelbox
from panelbox.datasets import load_dataset
from panelbox.diagnostics.unit_root import breitung_test, hadri_test, panel_unit_root_test
from panelbox.validation.unit_root import IPSTest, LLCTest

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from unit_root_helpers import (
    compare_unit_root_tests,
    interpret_results,
    plot_levels_vs_differences,
    recommend_transformation,
)

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "unit_root"
TABLES_DIR = OUTPUT_DIR / "tables" / "unit_root"
RESULTS_DIR = OUTPUT_DIR / "results" / "unit_root"

for d in [FIGURES_DIR, TABLES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

---

## Section 1: What is a Unit Root?

### The AR(1) Process

The simplest way to understand unit roots is through the **autoregressive process of order 1** (AR(1)):

$$y_t = \rho \cdot y_{t-1} + \varepsilon_t, \quad \varepsilon_t \sim \text{i.i.d.}(0, \sigma^2)$$

The behavior of this process depends entirely on the value of $\rho$:

| $\rho$ | Behavior | Variance | Name |
|--------|----------|----------|------|
| $|\rho| < 1$ | Mean-reverting | Bounded: $\frac{\sigma^2}{1-\rho^2}$ | **Stationary** (I(0)) |
| $\rho = 1$ | Random walk | Grows linearly: $t \cdot \sigma^2$ | **Unit root** (I(1)) |
| $|\rho| > 1$ | Explosive | Grows exponentially | Explosive (rare in economics) |

### Why "Unit Root"?

The characteristic equation of $y_t = \rho \cdot y_{t-1} + \varepsilon_t$ is $1 - \rho z = 0$, which has root $z = 1/\rho$. When $\rho = 1$, the root equals **unity** (1) -- hence "unit root."

### Key Insight

For a unit root process ($\rho = 1$), the variance $\text{Var}(y_t) = t \cdot \sigma^2$ grows without bound. This means:
- **Shocks are permanent**: past innovations never decay
- **No natural mean**: the series wanders without returning to any fixed level
- **Standard inference fails**: t-statistics do not follow the t-distribution

In [ ]:
np.random.seed(42)
T = 200


def simulate_ar1(T, rho, sigma=1.0):
    y = np.zeros(T)
    for t in range(1, T):
        y[t] = rho * y[t - 1] + np.random.normal(0, sigma)
    return y


y_stationary = simulate_ar1(T, rho=0.5)
y_near_unit = simulate_ar1(T, rho=0.95)
y_unit_root = simulate_ar1(T, rho=1.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(y_stationary, linewidth=0.8, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title(r"$\rho = 0.5$ (Stationary)", fontweight="bold")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("$y_t$")

axes[1].plot(y_near_unit, linewidth=0.8, color="darkorange")
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].set_title(r"$\rho = 0.95$ (Near Unit Root)", fontweight="bold")
axes[1].set_xlabel("Time")

axes[2].plot(y_unit_root, linewidth=0.8, color="forestgreen")
axes[2].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[2].set_title(r"$\rho = 1.0$ (Unit Root / Random Walk)", fontweight="bold")
axes[2].set_xlabel("Time")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_ar1_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Key observations:")
print(f"  rho=0.5: Std dev = {y_stationary.std():.2f} (bounded)")
print(f"  rho=0.95: Std dev = {y_near_unit.std():.2f} (large but bounded)")
print(f"  rho=1.0: Std dev = {y_unit_root.std():.2f} (grows with T)")

In [ ]:
from statsmodels.tsa.stattools import acf

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, y_series, _rho_val, title in zip(
    axes,
    [y_stationary, y_near_unit, y_unit_root],
    [0.5, 0.95, 1.0],
    [r"$\rho = 0.5$", r"$\rho = 0.95$", r"$\rho = 1.0$"],
):
    acf_vals = acf(y_series, nlags=30)
    ax.bar(range(len(acf_vals)), acf_vals, color="steelblue", alpha=0.7)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axhline(1.96 / np.sqrt(T), color="red", linestyle="--", alpha=0.5)
    ax.axhline(-1.96 / np.sqrt(T), color="red", linestyle="--", alpha=0.5)
    ax.set_title(f"ACF: {title}", fontweight="bold")
    ax.set_xlabel("Lag")
    ax.set_ylabel("Autocorrelation")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_acf_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("ACF interpretation:")
print("  Stationary (rho=0.5): ACF decays rapidly to zero")
print("  Near unit root (rho=0.95): ACF decays slowly")
print("  Unit root (rho=1.0): ACF barely decays - persistence")

---

### The Spurious Regression Problem

**Granger & Newbold (1974)** demonstrated a critical danger: regressing one I(1) series on another **independent** I(1) series produces:
- High $R^2$ (often > 0.5)
- Highly significant t-statistics
- **No real relationship** -- purely an artifact of common trends

This occurs because both random walks tend to drift in similar directions by chance, creating the illusion of a relationship. The Durbin-Watson statistic is typically very low in these spurious regressions.

**Rule of thumb (Granger & Newbold):** If $R^2 > DW$, suspect spurious regression.

In [ ]:
np.random.seed(42)
n = 200
x_rw = np.cumsum(np.random.randn(n))
y_rw = np.cumsum(np.random.randn(n))

model_spur = sm.OLS(y_rw, sm.add_constant(x_rw)).fit()
print("Spurious Regression: Two Independent Random Walks")
print("=" * 60)
print(f"R-squared: {model_spur.rsquared:.4f}")
print(f"Coefficient on x: {model_spur.params[1]:.4f}")
print(f"t-statistic: {model_spur.tvalues[1]:.4f}")
print(f"p-value: {model_spur.pvalues[1]:.4f}")
print(f"Durbin-Watson: {sm.stats.durbin_watson(model_spur.resid):.4f}")
print()
if model_spur.pvalues[1] < 0.05:
    print("WARNING: x and y are INDEPENDENT random walks,")
    print('but OLS finds a "significant" relationship!')
    print("This is the spurious regression problem.")
    print()
    print(
        f"Notice: R-squared ({model_spur.rsquared:.3f}) > DW ({sm.stats.durbin_watson(model_spur.resid):.3f})"
    )
    print("This is the Granger-Newbold rule-of-thumb for spurious regression.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_rw, label="x (random walk)", linewidth=1.5)
axes[0].plot(y_rw, label="y (random walk)", linewidth=1.5)
axes[0].set_title("Two Independent Random Walks", fontweight="bold")
axes[0].legend()
axes[0].set_xlabel("Time")

axes[1].scatter(x_rw, y_rw, alpha=0.4, s=20)
axes[1].plot(x_rw, model_spur.predict(), color="red", linewidth=2, label="OLS fit")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title(f"Spurious Regression ($R^2$ = {model_spur.rsquared:.3f})", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_spurious_regression.png", dpi=300, bbox_inches="tight")
plt.show()

---

### Panel Unit Root Tests vs. Univariate Tests

Classical unit root tests (ADF, Phillips-Perron, KPSS) test **one series at a time**. Panel unit root tests exploit the **cross-sectional dimension** to gain statistical power.

**Advantages of panel unit root tests:**
1. **More power**: Combining N cross-sections dramatically increases power over single-series tests
2. **Shorter T required**: Even with short time spans, the cross-sectional dimension compensates
3. **Heterogeneity**: Some tests (IPS, Fisher) allow different autoregressive coefficients $\rho_i$ across entities

**Trade-off:** Panel tests assume **cross-sectional independence** (or use corrections for dependence). If entities are strongly correlated (e.g., countries in the same region), this assumption may be violated, leading to size distortions.

---

## Section 2: Panel Unit Root Tests

We now present the four major panel unit root tests available in PanelBox. Each has different assumptions and null hypotheses.

### 2.1 Levin-Lin-Chu (LLC) Test

**Reference:** Levin, Lin & Chu (2002), *Journal of Econometrics*

**Model:**
$$\Delta y_{it} = \alpha_i + \rho \, y_{it-1} + \sum_{j=1}^{p_i} \gamma_{ij} \, \Delta y_{it-j} + \varepsilon_{it}$$

**Key assumption:** Common autoregressive coefficient $\rho$ across all entities.

| | |
|---|---|
| **H0** | $\rho = 0$ (all panels contain unit roots) |
| **H1** | $\rho < 0$ (all panels are stationary) |
| **Assumption** | Homogeneous $\rho$ across entities |
| **Test statistic** | Adjusted t-statistic $\rightarrow N(0,1)$ |

**When to use:** When you believe all entities share the same persistence parameter. Works best for homogeneous panels (e.g., G7 countries, similar industries).

**Limitation:** If some entities are stationary and others are not, LLC has low power because it imposes a common $\rho$.

### 2.2 Im-Pesaran-Shin (IPS) Test

**Reference:** Im, Pesaran & Shin (2003), *Journal of Econometrics*

**Model:**
$$\Delta y_{it} = \alpha_i + \rho_i \, y_{it-1} + \sum_{j=1}^{p_i} \gamma_{ij} \, \Delta y_{it-j} + \varepsilon_{it}$$

**Key feature:** Heterogeneous $\rho_i$ -- each entity can have a different persistence.

| | |
|---|---|
| **H0** | $\rho_i = 0$ for **all** $i$ (all panels have unit roots) |
| **H1** | $\rho_i < 0$ for **some** $i$ (at least some panels are stationary) |
| **Assumption** | Heterogeneous $\rho_i$ allowed |
| **Test statistic** | $\bar{t} = \frac{1}{N} \sum_{i=1}^{N} t_{\rho_i}$ standardized $\rightarrow N(0,1)$ |

**When to use:** The **default recommendation** for most applications. Allows heterogeneity in persistence across entities, which is realistic for most economic panels.

**Interpretation:** Rejecting H0 means **at least a fraction** of the entities are stationary -- not necessarily all of them.

### 2.3 Breitung (2000) Test

**Reference:** Breitung (2000), in Baltagi (ed.), *Nonstationary Panels*

**Key feature:** Uses a bias-corrected approach that is robust to deterministic trends.

| | |
|---|---|
| **H0** | All panels contain unit roots |
| **H1** | All panels are stationary |
| **Assumption** | Common $\rho$ (like LLC) |
| **Test statistic** | Modified t-ratio $\rightarrow N(0,1)$ |

**When to use:** When you suspect deterministic trends and want a test that is less sensitive to their specification. Complements LLC and IPS.

### 2.4 Hadri (2000) LM Test

**Reference:** Hadri (2000), *Econometrics Journal*

**CRITICAL DIFFERENCE: Reversed null hypothesis!**

| | |
|---|---|
| **H0** | All series are **STATIONARY** |
| **H1** | Some series contain unit roots |
| **Test statistic** | LM statistic $\rightarrow N(0,1)$ |

This is the **panel analog of the KPSS test**. While IPS/LLC/Breitung test *against* stationarity, Hadri tests *for* stationarity.

**Interpretation guide:**

| Hadri rejects H0? | Meaning |
|---|---|
| **Yes** (p < 0.05) | Evidence **against** stationarity (unit root likely) |
| **No** (p >= 0.05) | Consistent with stationarity |

**Why use both directions?** Combining tests with opposite null hypotheses gives a more complete picture:

| IPS/LLC/Breitung (H0: unit root) | Hadri (H0: stationary) | Conclusion |
|---|---|---|
| Fail to reject | Reject | **Strong evidence of unit root** |
| Reject | Fail to reject | **Strong evidence of stationarity** |
| Reject | Reject | Contradictory -- check specification |
| Fail to reject | Fail to reject | Inconclusive -- increase T or N |

---

## Section 3: Practical Application

Let us now apply these tests to real economic data. We will use the Penn World Table dataset, which contains macroeconomic variables for a panel of countries over several decades.

In [ ]:
pwt = load_dataset("penn_world_table", category="diagnostics")

print("Penn World Table (simulated)")
print("=" * 60)
print(f"Shape: {pwt.shape}")
print(f"Countries: {pwt['countrycode'].nunique()}")
print(f"Years: {pwt['year'].min()} - {pwt['year'].max()}")
print("\nFirst few rows:")
display(pwt.head(10))
print("\nSummary statistics:")
display(pwt.describe())

In [ ]:
# G7 countries for testing
g7 = ["USA", "GBR", "DEU", "FRA", "JPN", "ITA", "CAN"]
# Filter to G7 countries (they happen to be in the first 30 OECD codes)
# Check which are available
available = pwt["countrycode"].unique()
g7_available = [c for c in g7 if c in available]
print(f"Available G7 countries: {g7_available}")

# Use first 7 countries if G7 not all available
if len(g7_available) < 5:
    countries = list(available[:7])
    print(f"Using first 7 countries instead: {countries}")
else:
    countries = g7_available

df = pwt[pwt["countrycode"].isin(countries)].copy()

# Create log variables
df["log_gdp"] = np.log(df["rgdpna"])
df["log_capital"] = np.log(df["rkna"])
df["log_labor"] = np.log(df["emp"])
df["log_productivity"] = df["log_gdp"] - df["log_labor"]

print(f"\nWorking dataset: {df.shape[0]} observations")
print(f"Countries: {df['countrycode'].nunique()} ({', '.join(df['countrycode'].unique())})")
print(f"Years: {df['year'].min()} - {df['year'].max()}")
print("\nNew variables:")
display(
    df[["countrycode", "year", "log_gdp", "log_capital", "log_labor", "log_productivity"]].head()
)

### 3.1 Testing log(GDP) for Unit Roots

We expect real GDP (in logs) to be **I(1)** -- that is, it should contain a unit root. GDP typically grows over time without reverting to a fixed mean. The growth rate (first difference of log GDP) should be **I(0)** (stationary).

Let us apply each test individually to build intuition before using the unified interface.

In [ ]:
print("LLC Test on log(GDP)")
print("=" * 60)
try:
    llc = LLCTest(df, "log_gdp", "countrycode", "year", trend="ct")
    llc_result = llc.run()
    print("H0: All panels contain unit roots (common rho)")
    print("H1: All panels are stationary")
    print(f"\nStatistic: {llc_result.statistic:.4f}")
    print(f"P-value: {llc_result.pvalue:.4f}")
    print(f"Conclusion: {llc_result.conclusion}")
except Exception as e:
    print(f"LLC test error: {e}")

In [ ]:
print("IPS Test on log(GDP)")
print("=" * 60)
try:
    ips = IPSTest(df, "log_gdp", "countrycode", "year", trend="ct")
    ips_result = ips.run()
    print("H0: All panels contain unit roots (heterogeneous rho_i)")
    print("H1: Some panels are stationary")
    print(f"\nW-statistic: {ips_result.statistic:.4f}")
    print(f"t-bar: {ips_result.t_bar:.4f}")
    print(f"P-value: {ips_result.pvalue:.4f}")
    print(f"Conclusion: {ips_result.conclusion}")
except Exception as e:
    print(f"IPS test error: {e}")

In [ ]:
print("Breitung Test on log(GDP)")
print("=" * 60)
try:
    breit_result = breitung_test(df, "log_gdp", "countrycode", "year", trend="ct")
    print(breit_result.summary())
except Exception as e:
    print(f"Breitung test error: {e}")

In [ ]:
print("Hadri Test on log(GDP)")
print("=" * 60)
print("IMPORTANT: H0 is STATIONARITY (opposite of other tests!)")
print()
try:
    hadri_result = hadri_test(df, "log_gdp", "countrycode", "year", trend="ct")
    print(hadri_result.summary())
    print()
    if hadri_result.reject:
        print("Interpretation: REJECT stationarity -> evidence of unit root")
    else:
        print("Interpretation: FAIL TO REJECT stationarity -> consistent with stationarity")
except Exception as e:
    print(f"Hadri test error: {e}")

### Interpreting Results When Tests Agree and Disagree

The table below summarizes how to interpret combinations of results:

| IPS/LLC/Breitung (H0: unit root) | Hadri (H0: stationary) | Interpretation | Action |
|---|---|---|---|
| **Fail to reject** (p > 0.05) | **Reject** (p < 0.05) | Strong evidence of **unit root** | Use first differences |
| **Reject** (p < 0.05) | **Fail to reject** (p > 0.05) | Strong evidence of **stationarity** | Use levels |
| **Reject** | **Reject** | Contradictory | Check trend specification, sample size |
| **Fail to reject** | **Fail to reject** | Inconclusive | Increase N or T, try different lags |

**For log(GDP)**, we typically expect:
- IPS/LLC/Breitung: **Fail to reject** H0 (unit root present)
- Hadri: **Reject** H0 (not stationary)

This combination provides strong evidence that log(GDP) is I(1), and we should either work with first differences (GDP growth) or test for cointegration.

---

## Section 4: Unified Interface

PanelBox provides `panel_unit_root_test()` which runs multiple tests simultaneously and presents a comparative summary. This is the recommended approach for applied work.

In [ ]:
print("Unified Panel Unit Root Test: log(GDP)")
print("=" * 60)
try:
    gdp_result = panel_unit_root_test(
        data=df,
        variable="log_gdp",
        entity_col="countrycode",
        time_col="year",
        test="all",
        trend="ct",
    )
    print(gdp_result.summary_table())
except Exception as e:
    print(f"Unified test error: {e}")

In [ ]:
print("Comprehensive Comparison (all available tests)")
print("=" * 60)
comparison_df = compare_unit_root_tests(
    df,
    "log_gdp",
    "countrycode",
    "year",
    trend="ct",
    save_path=str(TABLES_DIR / "01_gdp_unit_root_tests.csv"),
)
display(comparison_df)

### 4.1 Testing GDP Growth -- Expected I(0)

If log(GDP) is I(1), then its first difference -- the growth rate -- should be I(0) (stationary). This is an important validation step: if the first difference is also non-stationary, the series might be I(2), which is unusual for economic variables.

In [ ]:
df["gdp_growth"] = df.groupby("countrycode")["log_gdp"].diff()
df_growth = df.dropna(subset=["gdp_growth"]).copy()

print("Unit Root Tests on GDP Growth (delta log(GDP))")
print("=" * 60)
growth_comparison = compare_unit_root_tests(
    df_growth,
    "gdp_growth",
    "countrycode",
    "year",
    trend="c",
    save_path=str(TABLES_DIR / "01_growth_unit_root_tests.csv"),
)
display(growth_comparison)

print("\nExpected: GDP growth should be stationary (I(0))")
print("  - IPS/LLC/Breitung should REJECT H0 (unit root)")
print("  - Hadri should FAIL TO REJECT H0 (stationarity)")

### 4.2 Testing Multiple Variables

In practice, you will test all variables in your model before running regressions. Variables that are I(1) require either differencing or cointegration analysis.

In [ ]:
variables = ["log_capital", "log_labor", "log_productivity", "labsh"]
var_results = {}

for var in variables:
    print(f"\nTesting: {var}")
    print("-" * 40)
    trend = "ct" if var != "labsh" else "c"
    try:
        var_results[var] = compare_unit_root_tests(df, var, "countrycode", "year", trend=trend)
        display(var_results[var])
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# Build summary table across all variables
summary_rows = []
for var in ["log_gdp", *variables]:
    trend = "ct" if var != "labsh" else "c"
    comp = compare_unit_root_tests(df, var, "countrycode", "year", trend=trend)

    row = {"Variable": var}
    for _, test_row in comp.iterrows():
        test_name = test_row["Test"].split("(")[0].strip()
        row[f"{test_name} p-val"] = test_row["P-value"]
        row[f"{test_name} reject"] = test_row["Reject_5pct"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("Summary of Unit Root Tests Across Variables")
print("=" * 80)
display(summary_df)
summary_df.to_csv(TABLES_DIR / "01_multi_variable_summary.csv", index=False)

---

## Section 5: Visualization

Visual inspection is a crucial first step before formal testing. Plotting series in levels and first differences helps build intuition about stationarity.

In [ ]:
fig = plot_levels_vs_differences(
    df,
    "log_gdp",
    "countrycode",
    "year",
    n_entities=4,
    save_path=str(FIGURES_DIR / "01_gdp_levels_vs_diff.png"),
)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, var, title in zip(
    axes.flatten(),
    ["log_gdp", "log_capital", "log_labor", "labsh"],
    ["log(GDP)", "log(Capital)", "log(Labor)", "Labor Share"],
):
    for country in countries[:4]:
        country_data = df[df["countrycode"] == country]
        ax.plot(country_data["year"], country_data[var], label=country, linewidth=1.2)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Panel Variables: Levels", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_panel_variables.png", dpi=300, bbox_inches="tight")
plt.show()

print("Visual check:")
print("  log_gdp, log_capital, log_labor: trending upward -> likely I(1)")
print("  labsh: fluctuating around mean -> likely I(0)")

---

## Section 6: Decision Making

After running the tests, you need to decide how to proceed with your analysis. Here is a decision flowchart:

### Decision Framework

1. **Run all four tests** on each variable (IPS, LLC, Breitung, Hadri)
2. **Check agreement:** Do all tests point in the same direction?
3. **If unanimous I(1):** Use first differences or proceed to cointegration testing
4. **If unanimous I(0):** Safe to use levels in panel regression
5. **If mixed results:**
   - Prioritize IPS (most flexible) and Hadri (opposite null)
   - Check if trend specification (`'c'` vs `'ct'`) affects results
   - Consider lag selection sensitivity
   - When in doubt, test both levels and first differences

### Trend Specification Guide

| `trend` | Deterministic terms | Use when |
|---|---|---|
| `'c'` | Constant only | Variable fluctuates around a level (e.g., interest rates, labor share) |
| `'ct'` | Constant + trend | Variable has a clear trend over time (e.g., GDP, capital stock) |

Choosing the wrong trend specification can severely distort results. When in doubt, use `'ct'` for macro variables with obvious trends.

In [ ]:
# Build results dict for interpretation
results_dict = {}
comp = compare_unit_root_tests(df, "log_gdp", "countrycode", "year", trend="ct")
for _, row in comp.iterrows():
    test_name = row["Test"].split("(")[0].strip()
    h0_type = "stationarity" if "Hadri" in row["Test"] else "unit_root"
    if not np.isnan(row["P-value"]):
        results_dict[test_name] = {"pvalue": row["P-value"], "h0": h0_type}

print("Interpretation of Results for log(GDP):")
print("=" * 60)
print(interpret_results(results_dict))
print(f"\nRecommendation: {recommend_transformation(results_dict)}")

### Implications for Modeling

The unit root test results have direct implications for your econometric approach:

**If variables are I(0) -- stationary:**
- Standard panel regression (fixed effects, random effects) is valid
- OLS estimates are consistent and inference is standard
- No need for differencing or cointegration

**If variables are I(1) -- non-stationary:**
- **Option 1: First differences.** Take $\Delta y_{it} = y_{it} - y_{it-1}$ for all I(1) variables. This eliminates the unit root but also removes long-run information.
- **Option 2: Cointegration.** If multiple I(1) variables share a common stochastic trend, they may be cointegrated. In this case, an error correction model (ECM) preserves long-run relationships while modeling short-run dynamics. See **Notebook 02: Cointegration Tests**.
- **Option 3: Panel ARDL.** Pesaran, Shin & Smith (1999) PMG/MG estimators can handle a mix of I(0) and I(1) variables.

**If variables have mixed orders:**
- Some I(0) and some I(1): ARDL/PMG approach is often appropriate
- Never regress I(0) on I(1) or vice versa without proper treatment

---

## Section 7: Key Takeaways

### Conceptual
1. A **unit root** means the series has no fixed mean and shocks have permanent effects
2. Regressing independent I(1) series produces **spurious regression** with misleadingly significant results
3. Panel unit root tests are more powerful than univariate tests because they exploit cross-sectional variation
4. The **Hadri test** has a reversed null hypothesis (H0: stationarity) compared to IPS/LLC/Breitung (H0: unit root)

### Practical
5. **Always test before modeling**: Run unit root tests on all variables before estimating panel regressions
6. **Use multiple tests**: No single test is best in all scenarios. Compare IPS (heterogeneous) with LLC (homogeneous) and confirm with Hadri (reversed null)
7. **Trend specification matters**: Use `trend='ct'` for trending variables (GDP, capital) and `trend='c'` for mean-reverting variables (inflation, labor share)
8. **First differences remove unit roots**: If $y_{it}$ is I(1), then $\Delta y_{it}$ is I(0)
9. **Cointegration preserves long-run info**: If variables are cointegrated, differencing loses valuable information -- use ECM instead

### Workflow
10. Visual inspection first (plot levels and differences)
11. Run all four tests via `compare_unit_root_tests()` or `panel_unit_root_test()`
12. Assess agreement across tests
13. Use `recommend_transformation()` for guidance
14. If I(1), proceed to cointegration testing (Notebook 02)

---

## Troubleshooting

### Common Errors and Solutions

**1. "Not enough time periods" error**
- Unit root tests require sufficient T (time periods) for each entity
- Minimum: T >= 10 for most tests, T >= 20 recommended
- Solution: Reduce the number of lags or use a longer time span

**2. Unbalanced panel issues**
- Some tests require balanced panels (same T for all entities)
- Solution: Balance the panel by restricting to common time periods, or use Fisher test which handles unbalanced panels

**3. Contradictory results across tests**
- This is common and does not mean something is wrong
- Check: Is the trend specification appropriate? Try both `'c'` and `'ct'`
- Check: Is the lag length adequate? Try different lag specifications
- Check: Is the panel balanced? Unbalanced panels can cause issues
- When in doubt: prioritize IPS (most general) and Hadri (opposite null)

**4. All tests fail to reject (inconclusive)**
- Panel may be too small (low N or T)
- Solution: Include more entities or time periods if available
- Consider: The variable may genuinely be near-integrated ($\rho$ close to but not equal to 1)

**5. Import errors**
- Ensure PanelBox is installed: `pip install panelbox`
- Some tests may not be available in older versions
- The unified `panel_unit_root_test()` with `test='all'` may skip unavailable tests gracefully

---

## Exercises

The following exercises allow you to practice applying panel unit root tests to different datasets. Solutions for Exercises 1-3 are available in `../solutions/`. Exercise 4 is independent (no solution provided).

### Exercise 1: Basic Unit Root Test (Easy)

**Task:** Test whether log(prices) has a unit root using the regional prices panel dataset.

**Steps:**
1. Load `prices_panel.csv` from the data directory
2. Run an IPS test on `log_price` with `trend='ct'`
3. Interpret the result: does the price index contain a unit root?
4. Test the first difference (`inflation`) -- is it stationary?

**Expected result:** Log prices should be I(1); inflation should be I(0).

In [ ]:
# Exercise 1: Basic Unit Root Test
# =================================
# Task: Test whether log(prices) has a unit root using the price panel dataset

# Step 1: Load the data
# prices = load_dataset("prices_panel", category="diagnostics")
# print(prices.head())

# Step 2: Run IPS test with trend='ct'
# YOUR CODE HERE

# Step 3: Interpret the result
# YOUR CODE HERE

# Step 4: Test first difference (inflation)
# YOUR CODE HERE

### Exercise 2: Comparing Tests (Medium)

**Task:** Compare how IPS and LLC behave with homogeneous vs. heterogeneous panels.

**Steps:**
1. Use the G7 subset (`df`) as the "homogeneous" panel
2. Use the full Penn World Table (`pwt`) as the "heterogeneous" panel with all 30 countries
3. Run both IPS and LLC on log(GDP) for each panel
4. Compare: Does LLC perform differently on the heterogeneous panel vs. IPS?

**Discussion:** When entities are heterogeneous (different growth patterns), LLC's assumption of a common $\rho$ may lead it to fail to reject H0 even when many entities are stationary.

In [ ]:
# Exercise 2: Comparing Tests
# ============================
# Task: Compare IPS vs LLC for G7 (similar) vs mixed panel (heterogeneous)

# Step 1: Create G7 panel (use 'countries' variable defined above)
# g7_data = df.copy()  # already filtered to G7-like countries

# Step 2: Create mixed panel with all 30 countries
# mixed_data = pwt.copy()
# mixed_data['log_gdp'] = np.log(mixed_data['rgdpna'])

# Step 3: Run IPS and LLC on each panel
# YOUR CODE HERE

# Step 4: Compare and discuss
# YOUR CODE HERE

### Exercise 3: Power Simulation (Hard)

**Task:** Simulate panel data with a known $\rho$ and measure the rejection rate (empirical power) of the IPS test.

**Steps:**
1. Write a function to simulate balanced panel AR(1) data with given $N$, $T$, and $\rho$
2. Simulate with $\rho = 0.95$ (near unit root), $N = 20$, $T = 50$
3. Repeat the simulation and test 100+ times, recording rejection rates
4. Create a power curve: plot rejection rate as a function of $\rho$ from 0.8 to 1.0

**Expected result:** Power should increase as $\rho$ moves away from 1.0, and should be near 5% (nominal size) when $\rho = 1.0$ exactly.

In [ ]:
# Exercise 3: Power Simulation
# ==============================
# Task: Simulate data with known rho, measure rejection rate

# Step 1: Function to simulate panel AR(1) data
# def simulate_panel_ar1(N, T, rho, sigma=1.0, seed=None):
#     """Simulate balanced panel AR(1) data."""
#     if seed is not None:
#         np.random.seed(seed)
#     records = []
#     for i in range(N):
#         y = np.zeros(T)
#         for t in range(1, T):
#             y[t] = rho * y[t-1] + np.random.normal(0, sigma)
#         for t in range(T):
#             records.append({'entity': i, 'time': t, 'y': y[t]})
#     return pd.DataFrame(records)

# Step 2: Simulate with rho=0.95 and test
# YOUR CODE HERE

# Step 3: Repeat 100+ times, compute rejection rates
# YOUR CODE HERE

# Step 4: Create power curve
# YOUR CODE HERE

### Exercise 4: Real Data Application (Independent -- No Solution Provided)

**Task:** Analyze state unemployment rates from `state_unemployment.csv`.

**Questions to answer:**
1. Is the unemployment rate stationary or I(1)? Test using all available methods.
2. Does excluding the 2008 recession period (2007-2009) change the results?
3. Is there a difference between testing `unemployment_rate` and `log_unemployment`?
4. What are the implications for modeling unemployment dynamics?

**Hint:** Unemployment rates are bounded between 0 and 100, so they cannot truly follow a random walk forever. However, they may appear to have a unit root over finite samples ("near-integration").

In [ ]:
# Exercise 4: Real Data Application (Independent - No Solution)
# ==============================================================
# Task: Analyze state unemployment rates

# Step 1: Load data
# unemp = load_dataset("state_unemployment", category="diagnostics")

# Step 2: Test full sample for unit roots
# YOUR CODE HERE

# Step 3: Test excluding 2008 recession
# YOUR CODE HERE

# Step 4: Discuss implications for modeling
# YOUR CODE HERE

---

## Next Steps

Now that you can test for unit roots in panel data, proceed to:

- **Notebook 02: Cointegration Tests** -- When I(1) variables share a long-run equilibrium relationship
- **Notebook 03: Cross-Sectional Dependence** -- Testing and correcting for dependence across entities
- **Notebook 04: Serial Correlation** -- Diagnosing autocorrelation in panel residuals
- **Notebook 05: Heteroskedasticity** -- Testing for non-constant variance

---

## References

### Seminal Papers
1. **Levin, A., Lin, C.-F., & Chu, C.-S. J.** (2002). Unit root tests in panel data: asymptotic and finite-sample properties. *Journal of Econometrics*, 108(1), 1-24.
2. **Im, K. S., Pesaran, M. H., & Shin, Y.** (2003). Testing for unit roots in heterogeneous panels. *Journal of Econometrics*, 115(1), 53-74.
3. **Breitung, J.** (2000). The local power of some unit root tests for panel data. In B. Baltagi (Ed.), *Nonstationary Panels, Panel Cointegration, and Dynamic Panels* (Advances in Econometrics, Vol. 15), pp. 161-178. JAI Press.
4. **Hadri, K.** (2000). Testing for stationarity in heterogeneous panel data. *The Econometrics Journal*, 3(2), 148-161.
5. **Maddala, G. S., & Wu, S.** (1999). A comparative study of unit root tests with panel data and a new simple test. *Oxford Bulletin of Economics and Statistics*, 61(S1), 631-652.
6. **Granger, C. W. J., & Newbold, P.** (1974). Spurious regressions in econometrics. *Journal of Econometrics*, 2(2), 111-120.

### Textbooks
7. **Baltagi, B. H.** (2021). *Econometric Analysis of Panel Data* (6th ed.). Springer.
8. **Hsiao, C.** (2022). *Analysis of Panel Data* (4th ed.). Cambridge University Press.
9. **Pesaran, M. H.** (2015). *Time Series and Panel Data Econometrics*. Oxford University Press.

### Survey Articles
10. **Breitung, J., & Pesaran, M. H.** (2008). Unit roots and cointegration in panels. In L. Matyas & P. Sevestre (Eds.), *The Econometrics of Panel Data* (3rd ed.), pp. 279-322. Springer.
11. **Banerjee, A.** (1999). Panel data unit roots and cointegration: an overview. *Oxford Bulletin of Economics and Statistics*, 61(S1), 607-629.